# MAIN-2: Annotation Divergence from GRCh38 Reference

For RBH gene pairs, characterise how pangenome assemblies diverge from GRCh38
reference gene models. Distinguish concordant divergence (both methods agree)
from method-specific divergence (annotation uncertainty).

**Categories:**
1. Both agree - same as reference
2. Both agree - diverged (high-confidence genuine variation)
3. Ensembl diverged, CAT agrees with reference (Ensembl-specific)
4. CAT diverged, Ensembl agrees with reference (CAT-specific)

**Input:** `intermediate_spreadsheets/divergence/` from workflow

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS_DIR = Path('../results')
DIV_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'divergence'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

COLORS = {
    'both_agree_reference': '#2ecc71',          # Green
    'both_agree_diverged': '#3498db',            # Blue (high-confidence variation)
    'ensembl_specific_divergence': '#e67e22',    # Orange
    'cat_specific_divergence': '#9b59b6',        # Purple
}
LABELS = {
    'both_agree_reference': 'Both agree\n(same as ref)',
    'both_agree_diverged': 'Both agree\n(diverged from ref)',
    'ensembl_specific_divergence': 'Ensembl-specific\ndivergence',
    'cat_specific_divergence': 'CAT-specific\ndivergence',
}

In [ ]:
# Load aggregated data
cross_tab = pd.read_csv(DIV_DIR / 'grch38_divergence_cross_tab.tsv', sep='\t')
per_asm = pd.read_csv(DIV_DIR / 'grch38_divergence_per_assembly.tsv', sep='\t')
by_biotype = pd.read_csv(DIV_DIR / 'grch38_divergence_by_biotype.tsv', sep='\t')

print(f"Cross-tabulation:")
display(cross_tab)
print(f"\nPer-assembly data: {len(per_asm)} assemblies")
print(f"Biotype breakdown: {len(by_biotype)} rows")

In [ ]:
# --- MAIN-2 Figure: 2x2 grouped bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1.5]})

# Panel A: Overall 2x2 cross-tabulation
ax = axes[0]
categories = cross_tab['divergence_category'].tolist()
pcts = cross_tab['pct_of_total'].tolist()
counts = cross_tab['total_gene_assembly_instances'].tolist()
colors = [COLORS.get(c, '#95a5a6') for c in categories]
labels = [LABELS.get(c, c) for c in categories]

bars = ax.bar(range(len(categories)), pcts, color=colors, edgecolor='white', width=0.7)
for bar, pct, count in zip(bars, pcts, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%\n(n={count:,})', ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Percentage of RBH gene pairs (%)', fontsize=11)
ax.set_title('A. Divergence categories', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel B: Biotype stratification
ax = axes[1]
top_biotypes = by_biotype.groupby('biotype')['count'].sum().nlargest(6).index.tolist()
bio_subset = by_biotype[by_biotype['biotype'].isin(top_biotypes)]

# Pivot for grouped bar chart
pivot = bio_subset.pivot_table(index='biotype', columns='divergence_category',
                                values='count', fill_value=0)
# Normalise to percentages within each biotype
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

cat_order = ['both_agree_reference', 'both_agree_diverged',
             'ensembl_specific_divergence', 'cat_specific_divergence']
cat_order = [c for c in cat_order if c in pivot_pct.columns]

x = np.arange(len(top_biotypes))
width = 0.18
for i, cat in enumerate(cat_order):
    if cat in pivot_pct.columns:
        vals = [pivot_pct.loc[bio, cat] if bio in pivot_pct.index else 0 for bio in top_biotypes]
        ax.bar(x + i * width, vals, width, color=COLORS.get(cat, '#95a5a6'),
               label=LABELS.get(cat, cat).replace('\n', ' '))

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(top_biotypes, fontsize=9, rotation=30, ha='right')
ax.set_ylabel('Percentage within biotype (%)', fontsize=11)
ax.set_title('B. Divergence by biotype', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main2_grch38_divergence.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_main2_grch38_divergence.pdf', bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'figure_main2_grch38_divergence.png'}")

In [ ]:
# --- Per-assembly distribution of concordant divergence ---
fig, ax = plt.subplots(figsize=(10, 5))

cols = ['pct_both_agree_reference', 'pct_both_agree_diverged',
        'pct_ensembl_specific', 'pct_cat_specific']
labels = ['Both agree\n(same as ref)', 'Both agree\n(diverged)', 'Ensembl-\nspecific', 'CAT-\nspecific']
cat_keys = ['both_agree_reference', 'both_agree_diverged',
            'ensembl_specific_divergence', 'cat_specific_divergence']

parts = ax.violinplot([per_asm[c].dropna() for c in cols], showmedians=True)
for i, (pc, key) in enumerate(zip(parts['bodies'], cat_keys)):
    pc.set_facecolor(COLORS[key])
    pc.set_alpha(0.7)

ax.set_xticks(range(1, len(labels)+1))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Percentage per assembly (%)', fontsize=11)
ax.set_title('Per-assembly distribution of GRCh38 divergence categories', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main2_per_assembly_divergence.png', dpi=300, bbox_inches='tight')
plt.show()